# 05. Rasters and cloud-ready files

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-terrio/blob/18574d4e7a8dabb99d3db65b20c6180185578482/notebooks/05-rasters-and-cogs.ipynb)



Reading/editing copy only in the shared Atlas kernel. TerrIO 0.9.0 uses a pyemscripten_2025_0 wasm wheel for Pyodide 0.29; the source pins jupyterlite-pyodide-kernel 0.8.0b0 (default Pyodide 0.29.4). This wheel is incompatible with Atlas Pyodide 314; do not install it here or downgrade the shared runtime. [Run this lesson in its matched source Lab](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=05-rasters-and-cogs.ipynb). The source uses Plotly 6.3.1, xarray 2025.10.1 and optional CDN/WebGL graphics. Natural Earth countries are real generalized geography; all other numerical fixtures are explicitly synthetic. Native HTTP range I/O, specialized drivers and real service/model calls remain optional external continuations.

**Provenance:** `notebooks/05-rasters-and-cogs.ipynb` at `18574d4e7a8dabb99d3db65b20c6180185578482`; original SHA-256 `3926185ec76f0a714813d6c87a8176418cc40596a6107cebcc9d89dd24c3dab2`. Original code MIT; original teaching prose, synthetic data and chapter SVG diagrams CC0. Natural Earth is public domain; TerrIO/dependency/EPSG terms remain separately scoped in retained notices. No TerrIO wheel redistributed.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Chapter 5: Rasters and cloud-ready files; abstract contour illustration](assets/05-rasters-and-cogs.svg)

# 05. Rasters and cloud-ready files

Pixels, missing values, and windows

**Learning goals:** Handle nodata; write a COG; compare a window with an array slice and resampled overview.

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays; earlier chapters are helpful but every notebook runs independently.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then **Run → Run All Cells**. First use downloads the runtime and packages. Native Python users install the repository requirements first. Plotly views need network access to its versioned JavaScript CDN and 3D views need WebGL; static figures and numerical summaries remain available.

[Open this chapter in JupyterLite](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=05-rasters-and-cogs.ipynb) · [Learning resources](https://jltobias.github.io/JupyterLite-terrio/book/resources.html)

**Data:** all generated measurements are synthetic. Natural Earth outlines, where used, are public domain. Generated files go to `outputs/`; download them to retain them outside this browser.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["terrio==0.9.0", "matplotlib", "plotly==6.3.1", "xarray==2025.10.1"])
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import terrio
from terrio_lab import (DATA, OUTPUT, CRS, TRANSFORM, CREDIT, terrain, raster,
                        countries, rings, map_outlines, image_grid, interactive, surface, globe)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready: TerrIO", terrio.__version__, "|", "browser" if sys.platform == "emscripten" else "native Python")

## Missing is not zero

We deliberately mask one corner. The sentinel -9999 is a storage convention, not a physical elevation. Means and colour limits must exclude it. Missingness may be spatially structured and can bias summaries.

In [ ]:
z = terrain(); z[:12, :18] = -9999
dem = raster(z)
valid = (z != dem.nodata) & np.isfinite(z)
print("Naive mean:", float(z.mean()), "valid mean:", float(z[valid].mean()))
print("Valid pixels:", int(valid.sum()), "/", z.size)
assert valid.sum() == z.size - 12*18
image_grid([z], ["Explicit nodata in the northwest"])
plt.show()

## Write a Cloud Optimized GeoTIFF

A COG arranges metadata and image blocks for efficient access. Saving a COG does not prove that a remote host supports range requests or CORS. This browser exercise reads a local file; the cloud chapter explains the network distinction.

In [ ]:
path = OUTPUT / "terrain-cog.tif"
dem.write(str(path), driver="COG", compress="DEFLATE")
cog = terrio.open_raster(str(path), lazy=True)
print("Size:", path.stat().st_size, "bytes; overviews:", cog.overviews)
tile = cog.read(1, window=(20, 30, 24, 16))
np.testing.assert_allclose(tile, z[30:46, 20:44])
assert cog.nodata == -9999
print("Window agrees with the original array")

## Resample deliberately

Nearest neighbour is appropriate for class labels. Averaging continuous elevations reduces detail but summarizes a footprint. Neither creates new information. The downsampled array has a different pixel size when stored as a raster.

In [ ]:
small = cog.read_resampled((24, 24), band=1, resampling="average")
small = np.asarray(small).squeeze()
image_grid([z, small], ["96 × 96 source", "24 × 24 average"])
plt.show()
assert small.shape == (24, 24)

## A byte budget

The uncompressed pixel budget is width × height × bands × bytes per value. File compression, caches, intermediate arrays, and browser runtime memory are separate costs. The graph compares logical array sizes, not measured network traffic.

In [ ]:
sizes = np.array([96, 512, 2048, 8192])
mib = sizes**2 * 4 / 1024**2
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(sizes.astype(str), mib, color="#008b8b")
ax.set(xlabel="Square image side (pixels)", ylabel="One float32 band (MiB)", yscale="log", title="Memory grows quadratically")
plt.show()

## Your investigation

Compare nearest and average resampling to 24 × 24. State which you would choose for a categorical land-cover map and why.

### Discussion guide

Choose nearest for class identifiers; averaging class codes produces values with no category meaning.

## Sources and credit

- [Source 1](https://terrio.dev/python/#rasters)
- [Source 2](https://www.ogc.org/standards/ogc-cloud-optimized-geotiff/)
- [Source 3](https://terrio.dev/remote/)

TerrIO: Qiusheng Wu, Mavrick Fitzgerald, Guillermo Morales Sanchez, and contributors (2026). *terrio: Pure-Rust geospatial raster, vector and point cloud data I/O*, v0.9.0. [Zenodo concept DOI](https://doi.org/10.5281/zenodo.23642204). The DOI identifies the project; the execution version is pinned separately.

Natural Earth: Tom Patterson, Nathaniel Vaughn Kelso, and contributors, v5.1.2, [public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/). Original explanations, synthetic data, code, and contour title card: JupyterLite-terrio contributors. Software and underlying third-party notices are documented in the repository's attribution page. Consult that page before redistributing dependency bundles.